# Name That LLM — G07 Technical Report

18-class source attribution · prompt-grouped development · two-stage LoRA · offline T4 inference

**Best verified Public BA: 0.674**. This is a CPU-only, self-contained report and aggregate-results check; it does not train models, load private inputs, generate predictions or submit to the competition.

[GitHub report](https://github.com/fjnuslw/name-that-llm) · [Competition](https://www.kaggle.com/competitions/name-that-llm)

Development OOF, Public scores and engineering samples have different evaluation scopes. No Private score, final rank, medal or independent unbiased evaluation is claimed.


# 方法与研究取舍

## 任务与数据

输入生成文本，输出 18 个指定来源模型之一；Balanced Accuracy 为 18 类 Recall 的均值。官方 2,699 条来自 150 个 Prompt，固定五折各 30 Prompt，同一题的文本不跨折。每个官方样本的 OOF 只来自未以其 Prompt 适配的折模型。

完整外部池为 Open Pool Kaggle V16 / open-pool-v16-r1，223,840 条、61 个归属来源。项目核查重复、跨标签同文、官方原文精确重合与已定位风险名单，保留来源和版本依据。导出保留 V16 发布文本，本地未额外删除；已知检查零命中不等于逐条模型身份或全部语义近题都已认证。四个官方派生匹配/风格字段仅保留溯源，不进入训练输入与采样权重。

选层使用按类及来源分层、无放回的固定 54k（每类 3,000）；正式训练使用完整外部池。54k 是计算预算选择，22.384 万也未被本项目受控收敛实验认证为最优量级。

## 冻结表征与可变层数搜索

两骨干为 Qwen3.8-27B（64 block，5,120 维）与 Qwen3.5-9B-Base（32 block，4,096 维）。原文右截断至 1,024 token，不加特殊 token 或聊天模板。对 final RMSNorm 前的 block 输出做 masked mean，逐样本逐层 L2，拼接并除以 sqrt(K)，接 18 类线性头。

先比较全部单层、末层、final norm、均匀八层、Top-8 与贪心方案，再搜索增层、删层与换层。003 的主候选预算为 27B 768、9B 384；少量候选用三个随机种子复核。

| 骨干 | 开发选择的层集合（1 起始） | 三种子 BA 均值 / SD |
|---|---|---|
| 27B | 19,22,33,34,47,48,49,50,51,52,53,54 | 0.480939 / 0.003586 |
| 9B | 6,14,17,18,19,20,21,22,23,24,32 | 0.458456 / 0.006375 |

最佳单层的三种子均值为 0.421662/0.395481。搜索复用同一开发 OOF，没有保留未触碰 Lockbox；这些是预算内候选，不是全局最优或 LoRA 后最优证明。

## 两阶段 LoRA 与 matched frozen 对照

两臂共用初始化、行、逻辑 batch、头优化配方与轮数。每臂先完整外部语料 1 epoch，再从外部检查点独立适配五个官方训练折各 2 epochs，另导出全官方 final。BF16 训练；LoRA r=16、alpha=32、dropout=0，覆盖选中 block 实际存在的 nn.Linear；逻辑 batch 16。

外部/官方阶段 LoRA 学习率为 1e-4/2e-5，头学习率 5e-4/2e-4；AdamW、weight decay 0.01、3% warmup 与梯度裁剪 1。根据读出梯度路径选择适配模块：27B 截至 54 层，放在 57–64 层的适配器无法影响当前分类损失；真实训练检查各块梯度和参数更新。

005 为 9B 拟合自有 centered Ridge 与概率尺度。每外折训练 Prompt 内分 80%/20% 初拟合/校准，再以全部外折训练组重拟合 centered 头。外折验证标签不拟合对应头与校准；内部校准组已被对应 LoRA/plain 头见过，内部成绩不是独立验证。

## 公共集成与自有分支

P0 保留 Omer V1 及其署名上游的 Transformer 多适配器、plain/twin/Gaussian、TF-IDF/SVC、风格统计、互补栈与条件 ATLAS。它们属于第三方复用。自有部分为数据版本治理、选层实验、27B/9B 训练对照、自有轻量头/校准拟合、分支替换、推理接入与效率验证。

自有 27B final 和 9B 五折分别完整替换原两个尾部分支，避免对旧完整预测重复叠加。先 P0 与 27B 算术概率混合，再每折加入 9B 校准 log-prob；执行原共同后处理后平均。具体公式见仓库 README 与融合示例。


In [ ]:
import json, math, hashlib, csv
from pathlib import Path

METRICS = json.loads('{"format": "name-that-llm-public-report-v1", "snapshot_date": "2026-11-16", "task": {"classes": 18, "official_rows": 2699, "prompt_groups": 150, "cv_folds": 5}, "data": {"release": "open-pool-v16-r1", "kaggle_version": 16, "external_rows": 223840, "credited_sources": 61, "selection_rows": 54000, "selection_per_class": 3000}, "layer_selection": {"qwen38-27b": {"layers": [19, 22, 33, 34, 47, 48, 49, 50, 51, 52, 53, 54], "development_ba_mean": 0.48093876874637503, "seed_sd": 0.0035855538951600797, "seeds": [{"seed": 42, "ba": 0.4860054685558042}, {"seed": 20261112, "ba": 0.4785806611981109}, {"seed": 20261113, "ba": 0.47823017648521}]}, "qwen35-9b-base": {"layers": [6, 14, 17, 18, 19, 20, 21, 22, 23, 24, 32], "development_ba_mean": 0.4584563758389262, "seed_sd": 0.006375094139605285, "seeds": [{"seed": 42, "ba": 0.4674695500869997}, {"seed": 20261112, "ba": 0.45376336067611234}, {"seed": 20261113, "ba": 0.4541362167536664}]}}, "matched_development_oof": [{"model": "qwen38-27b", "arm": "frozen", "ba": 0.4726820780512056}, {"model": "qwen38-27b", "arm": "lora", "ba": 0.6509321401938851}, {"model": "qwen35-9b-base", "arm": "frozen", "ba": 0.4426696495152871}, {"model": "qwen35-9b-base", "arm": "lora", "ba": 0.6309072831220482}], "readout_development_oof": {"plain": 0.6305369127516779, "centered": 0.6078846631866767, "calibrated": 0.6712652249565001}, "public_scores": [{"experiment": "V3", "a27": 0.12, "b9": 0.4, "public_ba": 0.669, "submission_ref": 58339923}, {"experiment": "G01", "a27": 0.25, "b9": 0.4, "public_ba": 0.666, "submission_ref": 58363608}, {"experiment": "G04", "a27": 0.4, "b9": 0.4, "public_ba": 0.672, "submission_ref": 58363612}, {"experiment": "G07", "a27": 0.55, "b9": 0.4, "public_ba": 0.674, "submission_ref": 58364991}, {"experiment": "A27", "a27": null, "b9": null, "public_ba": 0.64, "submission_ref": 58365339}, {"experiment": "A09", "a27": null, "b9": null, "public_ba": 0.626, "submission_ref": 58365344}], "inference_180_rows": {"27b": {"rows": 180, "members": 1, "before_forward_seconds": 158.23564515600003, "after_forward_seconds": 103.80736635100004, "time_reduction": 0.34396977211639446}, "9b": {"rows": 180, "members": 5, "before_forward_seconds": 260.562636258, "after_forward_seconds": 181.25759484699995, "time_reduction": 0.30436075774300564}}, "008_status_checked_utc": "2026-11-16T09:51:30.886628+00:00", "008_status": [{"experiment": "N01", "status": "error", "submission_status": "SubmissionStatus.COMPLETE", "public_ba": null, "runtime_timeout": true}, {"experiment": "N02", "status": "submitted", "submission_status": "SubmissionStatus.PENDING", "public_ba": null, "runtime_timeout": false}, {"experiment": "N03", "status": "submitted", "submission_status": "SubmissionStatus.PENDING", "public_ba": null, "runtime_timeout": false}], "008_example_runtime": [{"run": "shared-v1", "rows": 3, "selected_rows": 1, "total_seconds": 12958.808849583, "expert_seconds": 10181.390401321001, "all_25_segments": true}, {"run": "N02-v2", "rows": 3, "selected_rows": 1, "total_seconds": 12176.431473701, "expert_seconds": 9648.870746494, "all_25_segments": true}, {"run": "N03-v1", "rows": 3, "selected_rows": 1, "total_seconds": 11805.980533963, "expert_seconds": 8819.447452305001, "all_25_segments": true}], "evaluation_boundary": {"oof_is_development": true, "layer_selection_reused_development": true, "inner_calibration_groups_seen_by_lora": true, "independent_lockbox_remains": false, "private_score_available": false, "hidden_runtime_available": false, "without_p0_dual_model_public_control_available": false}}')
EVIDENCE_MANIFEST = json.loads('{"format": "name-that-llm-evidence-manifest-v1", "metrics_sha256_canonical_json": "3648407ad220e9b5f7770c11e0a8deb4c4c808b77e8b0d039129ad4766f31d04", "archive_fingerprints": [{"stage": "003", "archive": "20261112T091407Z.zip", "sha256": "355938cab8c9eab18c3a1ded0551fb98b7d1493d248c3eddabedbf247fc2d4ca"}, {"stage": "004", "archive": "20261113T065343Z-parallel.zip", "sha256": "cdd70a46f0830c2e1e46ea9285ece750114b1eb5f10727d3af8fd270d64c53a8"}, {"stage": "005", "archive": "20261113T171219Z.zip", "sha256": "7a7c81cff6ad53451e7eba1ac1d566e423321f37ca52630e70f088b4a9183af0"}], "score_records": [{"experiment": "V3", "a27": 0.12, "b9": 0.4, "public_ba": 0.669, "submission_ref": 58339923}, {"experiment": "G01", "a27": 0.25, "b9": 0.4, "public_ba": 0.666, "submission_ref": 58363608}, {"experiment": "G04", "a27": 0.4, "b9": 0.4, "public_ba": 0.672, "submission_ref": 58363612}, {"experiment": "G07", "a27": 0.55, "b9": 0.4, "public_ba": 0.674, "submission_ref": 58364991}, {"experiment": "A27", "a27": null, "b9": null, "public_ba": 0.64, "submission_ref": 58365339}, {"experiment": "A09", "a27": null, "b9": null, "public_ba": 0.626, "submission_ref": 58365344}], "validation": "20 key OOF files recomputed locally; ID, label, Prompt and fixed-fold alignment checked", "shared_payload": "aggregates, reporting code and dependency references", "raw_text_ids_predictions_weights_or_credentials_included": false}')
print('Snapshot:', METRICS['snapshot_date'])
print('Task:', METRICS['task'])
print('Data:', METRICS['data'])


# 实验与评价边界

以下数值来自本地原始 OOF 回传与官方评分回执。公开包仅包含汇总；原始 20 份关键 OOF 已在本地重新计算 argmax、18 类 Recall 均值，并核对 ID/标签/Prompt/固定折号。回传 ZIP 指纹位于 results/evidence-manifest.json。

## 同条件 LoRA 对照（开发 OOF）

| 骨干 | Frozen | LoRA | BA 差值 |
|---|---:|---:|---:|
| 27B | 0.472682 | 0.650932 | +0.178250 |
| 9B | 0.442670 | 0.630907 | +0.188238 |

003 选层与 004 训练使用的数据量和头配方不同，不能直接作差归因 LoRA。004 的 matched 比较固定了这些条件，但先前架构选择仍依赖同一开发集。004 双模型融合诊断为 0.657604，与 G07 不是同一部署配方。

005 9B plain/centered/calibrated 开发 OOF 为 0.630537/0.607885/0.671265。内部校准组被对应 LoRA/plain 见过，不能把该分数称为全流程无偏嵌套验证。

## 已得分 Public 对照

| 版本 | 第一阶段 P0/27B | 第二阶段 9B log share | Public BA |
|---|---|---:|---:|
| V3 | 88/12 | 40% | 0.669 |
| G01 | 75/25 | 40% | 0.666 |
| G04 | 60/40 | 40% | 0.672 |
| G07 | 45/55 | 40% | **0.674** |
| A27 | 自有 27B 单分支 | — | 0.640 |
| A09 | 自有 9B 五折 | — | 0.626 |

G07 比 V3 高 0.005 BA，即 0.5 个百分点。完整权重曲线不是全程单调；未测更高占比，不称已找到最优点。单分支保留校准与共同后处理；缺少“27B+9B，无 P0”的对照，因此不能将完整系统对单分支的全部差值归因 P0。无隐藏逐行标签，不声称小差异显著或 Private 必然提高。

## 最有信息量的后续实验（未完成）

固定资产、输入与后处理，分别检验无 P0 的自有双模型、降低 9B 权重、移除 centered 混合、单独升级 P0 和单独增加 GLM 专家。现有缺失对照按未测处理，不填入臆测成绩。


In [ ]:
scores = {r['experiment']: r['public_ba'] for r in METRICS['public_scores']}
for r in METRICS['public_scores']:
    print(f"{r['experiment']:>3}  Public BA={r['public_ba']:.3f}  submission={r['submission_ref']}")
delta = scores['G07'] - scores['V3']
assert math.isclose(delta, .005, abs_tol=1e-12)
print(f"G07 vs V3: +{delta:.3f} BA = +{100*delta:.1f} percentage points")
print("No-P0 dual-model control: not measured")


In [ ]:
for model in ('qwen38-27b', 'qwen35-9b-base'):
    arms = {r['arm']: r['ba'] for r in METRICS['matched_development_oof'] if r['model']==model}
    print(model, 'matched development OOF:', arms, 'delta:', arms['lora']-arms['frozen'])
for branch, r in METRICS['inference_180_rows'].items():
    reduction = 1-r['after_forward_seconds']/r['before_forward_seconds']
    assert math.isclose(reduction, r['time_reduction'], abs_tol=1e-12)
    print(f"{branch}: same 180 rows, {r['members']} member(s), forward time reduction {100*reduction:.1f}%")
print("Scope: branch forward time only, not full-system/hidden runtime")


# 双 T4 推理与失败复盘

## 已落地工程

A800 训练侧采用完整骨干副本与显式 LoRA/头梯度同步，保持逻辑 batch；保存优化器、RNG、更新位置与样本计数。没有已核实的通用多卡线性加速倍数。

双 T4 断网推理中，27B 采用 NF4 权重与 FP16 计算，9B 采用 FP16；加载一次 9B 基座并切换五折 adapter。固定模型 revision、tokenizer 配置和类别顺序；使用离线 wheelhouse、顺序基座暂存、子进程释放显存，以及 ID/概率/Parquet 契约检查。

同 180 行、对应未见 Prompt 折的 9B plain 输出相对服务器 BF16，NF4 的 argmax 一致率 90.56%，FP16 99.44%，支持选用 FP16。这不是全系统 BA 或全部文本无损证明；27B 也没有同 final 的完整 BF16/NF4 等价证明。

| 分支（同一 180 行） | 原设置 | 长度组批设置 | 前向耗时 | 减少 |
|---|---|---|---|---:|
| 27B final | NF4，batch 4 | 最多 16 行，4096 padded token | 158.24→103.81 秒 | 34.4% |
| 9B 五折合计 | FP16，batch 8 | 最多 32 行，4096 padded token | 260.56→181.26 秒 | 30.4% |

排序后恢复 ID 顺序。plain argmax 分别 180/180、900/900 一致；9B 校准后 899/900，固定融合 180/180 一致。比例只对应分支前向，未包含所有加载、公共上游或隐藏集。部分 GPU reserved 统计于加载后重置，只覆盖前向，不能标为全程显存峰值。

V3 三行全流程约 43.47 分钟，G07 约 43.29 分钟；正式得分说明对应版本完成过平台隐藏重跑，但隐藏阶段耗时不可见。

## 008：真实 GLM 似然专家

该实验把 P0 一个 14B 子分支改为 Nuri v4/Nishant 两个 adapter 实际前向后的 72:28 读出混合，并在完整自有融合及原后处理之后接真实 GLM5.3/Flash 的 teacher-forced 似然证据。

复用公开的 54 维特征与 18 类 LogisticRegression 状态，未自训此专家。常规最多覆盖 20% 文本，每段 160 token，受 130k 估算 token-row 预算约束；极小输入至少 1 行，三行样例实际处理一行。全部 6 个 Flash、19 个 GLM 分片均执行。

三行全流程为 3.28–3.60 小时，专家部分为 2.45–2.83 小时。状态快照时间（UTC）：2026-11-16T09:51:30.886628+00:00。

N01: hidden runtime timeout, score=None; N02: SubmissionStatus.PENDING, score=None; N03: SubmissionStatus.PENDING, score=None

小样例有效不等于完整规模 9 小时内通过。加载/分片 I/O、固定成本与 token 前向需要一起预算；日志尚不能精确分解隐藏超时根因。该分支作为失败与局限展示，不能称稳定部署优化。


## G07 outer-fusion interface

The next cell shares the standalone formula. It requires NumPy only when imported and external aligned probabilities/postprocessing. No learned assets or full public pipeline are included.

In [ ]:
FUSION_SOURCE = '"""G07 outer fusion. Public-parent/postprocessing implementations are external.\n\nFormula follows the credited Omer V1 pipeline. This standalone interface\nexposes the project\'s configurable outer shares; it does not ship trained assets.\n"""\nimport numpy as np\n\ndef softmax(x):\n    x = np.asarray(x, dtype=np.float64)\n    x = x - x.max(axis=-1, keepdims=True)\n    e = np.exp(x)\n    return e / e.sum(axis=-1, keepdims=True)\n\ndef fuse(parent, own27, own9_logp, postprocess, a27=.55, b9=.4):\n    """postprocess receives N x 18 probabilities and returns probabilities."""\n    parent = np.asarray(parent, dtype=np.float64)\n    own27 = np.asarray(own27, dtype=np.float64)\n    own9_logp = np.asarray(own9_logp, dtype=np.float64)\n    if not (0 <= a27 <= 1 and 0 <= b9 < 1):\n        raise ValueError("invalid shares")\n    if parent.ndim != 2 or parent.shape[1] != 18:\n        raise ValueError("parent must be N x 18")\n    if own27.shape != parent.shape or own9_logp.shape != (5, *parent.shape):\n        raise ValueError("unaligned branch shapes")\n    for p in (parent, own27, np.exp(own9_logp)):\n        if not np.isfinite(p).all() or (p < 0).any() or not np.allclose(p.sum(-1), 1, atol=1e-6):\n            raise ValueError("invalid probabilities")\n    p1 = (1-a27)*parent + a27*own27\n    output = []\n    for lp in own9_logp:\n        q = softmax((1-b9)*np.log(np.maximum(p1, 1e-15)) + b9*lp)\n        p = np.asarray(postprocess(q), dtype=np.float64)\n        if p.shape != parent.shape or not np.isfinite(p).all() or (p < 0).any() or not np.allclose(p.sum(-1), 1, atol=1e-6):\n            raise ValueError("postprocess must return aligned probabilities")\n        output.append(p)\n    return np.mean(output, axis=0)\n'
compile(FUSION_SOURCE, 'fusion.py', 'exec')
print(FUSION_SOURCE)


# 证据、复用与公开边界

本报告以 2026-11-16 的本地证据快照为基础；Public 分数回执核对日期为 2026-11-15。OOF 为开发诊断，Public 为平台隐藏重跑结果，三行/180 行为工程验证，各自分开记录。

## 已复核

- 官方 2,699 行、150 Prompt、18 类；固定 CV 五折各 30 Prompt。
- 54,000 个唯一外部 ID，每类 3,000。
- 20 份关键 OOF 的完整 ID、标签、Prompt、逐行固定折号、有限 logits、argmax 与 BA。
- 已得分版本 V3/G01/G04/G07/A27/A09 的本地官方 API 回执。
- 同输入分支效率与 008 实际执行审查记录。

公开包不含原始文本、逐行 ID/预测、训练权重、凭据或内部运维文件。指纹可用于获得合法原始归档后的身份核验；本包的轻量脚本只验证已公开汇总的完整性与算术，不声称重跑 OOF、训练或平台评分。

## 本赛分享要求

发布前通过官方 Competition API 读取本赛 rules、foundational-rules 与 Code Requirements。公开相关代码须兼顾第三方权利，并在本赛关联的 Kaggle Notebook 或讨论中向所有参赛者提供。本包通过自包含的关联 Notebook 公开同一报告和轻量代码，采用 Apache-2.0；不重新分发官方原文或第三方权重。

[本赛规则](https://www.kaggle.com/competitions/name-that-llm/rules) · [本赛说明](https://www.kaggle.com/competitions/name-that-llm)

## 第三方来源

公共 P0、Gaussian/twin、TF-IDF/风格分支、ATLAS 与共同后处理保留原始署名。项目贡献范围说明不自动认定为单人独立完成全部工作，也不认领上游预拟合模型为原创。具体来源见 THIRD-PARTY-NOTICES.md。

报告一致性摘要 SHA256：3648407ad220e9b5f7770c11e0a8deb4c4c808b77e8b0d039129ad4766f31d04


# Third-party sources and scope

This repository publishes newly written explanatory material, aggregate results
and lightweight reporting/fusion code. It does not redistribute third-party
training data, checkpoints, prefit heads, compiled binaries or full pipelines.
Their original licenses and attribution requirements continue to apply.

- Competition: https://www.kaggle.com/competitions/name-that-llm
- Open Pool Kaggle V16, crtproduction and credited source contributors:
  https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool/versions/16
- Qwen3.8-27B, fixed revision 37561816e66b5b9153b1b64c10968c18c1e6f4a0:
  https://huggingface.co/Qwen/Qwen3.8-27B/tree/37561816e66b5b9153b1b64c10968c18c1e6f4a0
- Qwen3.5-9B-Base, revision 68c46c4b3498877f3ef123c856ecfde50c39f404:
  https://huggingface.co/Qwen/Qwen3.5-9B-Base/tree/68c46c4b3498877f3ef123c856ecfde50c39f404
- Omer Faruk Merey, fixed V1, scriptVersionId 367805055:
  https://www.kaggle.com/code/omerfarukmerey/0-682-0-681-qwen3-8-27b-gsq-12-mix-0-50?scriptVersionId=367805055
  Retain the original upstream credits, including lightningv08, heliosli,
  mercurydust, Roman Tamrazov and other contributors.
- Public P0 assets also credit Dhruv Tibarewal, Nishant Dahal, Nuri Dudak,
  Lucas Boesen and the authors of the original model, encoder, style and SVC assets.
  A versioned reference inventory is provided in results/public-dependencies.json.
- ATLAS assets, josephadamski91, V3:
  https://www.kaggle.com/datasets/josephadamski91/ntl-atlas-qwen3-8b-v1/versions/3
- 008 likelihood extension: Lightning V52 / feiyu V1 and their upstream
  feature/classifier/execution contributors:
  https://www.kaggle.com/code/lightningv08/0-694-glm-5-3-likelihood-qwen3-8-27b-lora
  https://www.kaggle.com/code/feiyu26/ntl-0684-v1-graft?scriptVersionId=369151207
  This experiment is not presented as a trained or reliably deployed original expert.

The standalone fusion formula follows the credited arithmetic/log-probability
outer pipeline. Its callback intentionally leaves original common postprocessing
external; this is not the full scoring implementation. The repository's
Apache-2.0 license applies only to newly authored material supplied here.


In [ ]:
VERIFY_SOURCE = '"""Check the published evidence digest and arithmetic, without private data."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport math\n\nroot = Path(__file__).resolve().parents[1]\nm = json.loads((root / "results/metrics.json").read_text(encoding="utf-8"))\ne = json.loads((root / "results/evidence-manifest.json").read_text(encoding="utf-8"))\ncanonical = json.dumps(m, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\nassert hashlib.sha256(canonical.encode()).hexdigest() == e["metrics_sha256_canonical_json"]\nscores = {r["experiment"]: r["public_ba"] for r in m["public_scores"]}\nassert math.isclose(scores["G07"]-scores["V3"], .005, abs_tol=1e-12)\nassert max(scores.values()) == scores["G07"]\nassert all(0 <= r["ba"] <= 1 for r in m["matched_development_oof"])\nassert m["data"]["selection_per_class"] * m["task"]["classes"] == m["data"]["selection_rows"]\nfor branch, r in m["inference_180_rows"].items():\n    assert math.isclose(1-r["after_forward_seconds"]/r["before_forward_seconds"], r["time_reduction"], abs_tol=1e-12)\nprint("Published evidence digest and arithmetic: passed")\nprint("G07 - V3: +0.005 BA = +0.5 percentage points")\nprint("Scope: aggregate consistency; does not rerun training, OOF or hidden scoring")\n'
compile(VERIFY_SOURCE, 'verify_results.py', 'exec')
print(VERIFY_SOURCE)


In [ ]:
CANONICAL = json.dumps(METRICS, sort_keys=True, ensure_ascii=False, separators=(',', ':'))
assert hashlib.sha256(CANONICAL.encode()).hexdigest() == '3648407ad220e9b5f7770c11e0a8deb4c4c808b77e8b0d039129ad4766f31d04'
out = Path.cwd()
for folder in ('results','src','scripts'): (out/folder).mkdir(exist_ok=True)
(out/'results/metrics.json').write_text(json.dumps(METRICS, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
(out/'results/evidence-manifest.json').write_text(json.dumps(EVIDENCE_MANIFEST, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
(out/'src/fusion.py').write_text(FUSION_SOURCE, encoding='utf-8')
(out/'scripts/verify_results.py').write_text(VERIFY_SOURCE, encoding='utf-8')
print('Aggregate evidence digest: passed')
print('Exported results/, src/fusion.py and scripts/verify_results.py')
print('No training, weights, original texts, raw row predictions or competition submission')
